# Task 4

In [5]:
! pip install torch


[notice] A new release of pip is available: 23.2.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
! pip install transformers


[notice] A new release of pip is available: 23.2.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = "readerbench/RoGPT2-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)

C:\Users\User\Desktop\DM\github\DL4NLP_2026\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 149/149 [00:00<00:00, 8762.02it/s]
[transformers] GPT2LMHeadModel LOAD REPORT from: readerbench/RoGPT2-base
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...11}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [13]:
prompt = "Vremea de astazi este foarte"

In [14]:
inputs = tokenizer(prompt, return_tensors="pt")
input_ids = inputs["input_ids"]
print("Prompt:", prompt)
print("Token ids:", input_ids[0].tolist())
print("Tokens:", tokenizer.convert_ids_to_tokens(input_ids[0]))

Prompt: Vremea de astazi este foarte
Token ids: [54, 247, 1771, 253, 13960, 375, 737]
Tokens: ['V', 're', 'mea', 'Ġde', 'Ġastazi', 'Ġeste', 'Ġfoarte']


In [15]:
outputs = model.generate(
    **inputs,
    max_new_tokens=2,
    do_sample=False,
    pad_token_id=tokenizer.eos_token_id,  # GPT-2 has no pad token
)

In [16]:
new_ids = outputs[0, input_ids.shape[1]:]
print("\nGenerated token ids:", new_ids.tolist())
print("Generated tokens:", tokenizer.convert_ids_to_tokens(new_ids))
print("Continuation:", repr(tokenizer.decode(new_ids)))
print("Full text:", tokenizer.decode(outputs[0]))


Generated token ids: [4150, 12]
Generated tokens: ['Ġrece', ',']
Continuation: ' rece,'
Full text: Vremea de astazi este foarte rece,


In [17]:
import sys, os

sys.path.insert(0, os.path.abspath("../task2"))

from task2 import NGramModel, load_corpus, split_sentences, tokenize

corpus_path = os.path.abspath("../task2/corpus_ro.txt")
raw_text, _ = load_corpus(corpus_path, refresh=False)
sentences = split_sentences(raw_text)
bigram = NGramModel(2, sentences)

ro_tokens = tokenize(prompt)
context_tuple, top5 = bigram.predict_next(ro_tokens)

print(f"Prompt          : '{prompt}'")
print(f"Context used    : {context_tuple}")
print()
print("N-gram bigram top-5 predictions:")
for word, p in top5:
    print(f"  {word:<20} P(Laplace) = {p:.5f}")
print()
print(f"RoGPT-2 continuation: '{tokenizer.decode(new_ids).strip()}'")

Prompt          : 'Vremea de astazi este foarte'
Context used    : ('foarte',)

N-gram bigram top-5 predictions:
  mare                 P(Laplace) = 0.00025
  multe                P(Laplace) = 0.00020
  puține               P(Laplace) = 0.00020
  active               P(Laplace) = 0.00015
  mult                 P(Laplace) = 0.00015

RoGPT-2 continuation: 'rece,'


## Comparison: N-gram model vs. Neural language model

Both models receive the same Romanian prompt: "Vremea de astazi este foarte".

The N-gram bigram uses only the last word ("foarte") as context and ranks continuations by
co-occurrence counts from the Wikipedia corpus, applying Laplace smoothing for unseen pairs.
Its predictions are plausible Romanian words but reflect only surface-level statistics --
it cannot capture meaning or long-range dependencies.

RoGPT-2 is a GPT-2 model pretrained on a large Romanian corpus. It encodes the full prompt
through self-attention across all 5 tokens and generates a continuation that is both
grammatically correct and semantically coherent in Romanian.

The comparison shows the core trade-off: the N-gram model is simple, interpretable, and
requires little data, but is limited by sparse counts and a fixed context window.
RoGPT-2 generalises far better to unseen contexts thanks to distributed word representations
and deep contextual reasoning learned during pretraining.